### Static Linear Elasticity in 2D (Cartesian) — Analytic Verification

This notebook runs a small test suite of **static linear-elastic** problems in 2D Cartesian geometry.  
Material parameters are set to **E = 160 GPa**, **ν = 0.31**, with Lamé parameters **μ** and **λ** computed from (E, ν).
Each case is solved and **systematically compared to a closed-form (analytic) solution**.

**Test cases**
- **jdd0** — prescribed displacements on all boundaries, solution independant of $\lambda$
- **jdd1** — $\lambda=0$, body force + prescribed displacements (Dirichlet).
- **jdd1b** — $\lambda=0$, body force + prescribed tractions (Neumann) on boundaries.
- **jdd2** — full μ–λ formulation (λ ≠ 0).
- **jdd3** — self-weight loading: top edge fixed, zero tractions elsewhere.
- **jdd4** — μ–λ formulation with imposed tractions.

In [ ]:
from trustutils import run
import os, sys
run.reset()
run.initBuildDirectory()
run.addCase(".", "jdd0.data")
run.addCase(".", "jdd1.data")
run.addCase(".", "jdd1b.data")
run.addCase(".", "jdd2.data")
run.addCase(".", "jdd3.data")
E = 160e9
nu = 0.31
mu = E/(2*(1+nu))
lambda_ = E*nu/((1+nu)*(1-2*nu))
A = 1e-1
alpha, beta = 1e-1, 1e-1
run.addCaseFromTemplate("jdd4.data", "results", {"mu": mu, "lambda": lambda_, "E" : E, "nu" : nu, "A" : A})
run.runCases()
run.tablePerf()

In [ ]:
from trustutils import plot
import numpy as np

fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/jdd0.dt_ev", label=" ")
fig.scale("linear", "log")

fig = plot.Graph()
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd0_DX.son", marker="-x", label="TRUST", compo=1)
x = np.linspace(0, 1, 100)
sol = np.cosh(np.pi * x) * np.sin(np.pi * 0.5)
fig.add(x, sol, label="solution")

In [ ]:
fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/jdd1.dt_ev",  label="1")
fig.addResidu(f"{run.BUILD_DIRECTORY}/jdd1b.dt_ev", label="1b")
fig.scale("linear", "log")

fig = plot.Graph()
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd1_DX.son", marker="-x", label="TRUST-deplacement", compo=1)
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd1b_DX.son", marker="-x", label="TRUST-contrainte", compo=1)
x = np.linspace(0, 1, 100)
sol = np.sin(np.pi * 0.5) * np.cos(np.pi * x)
fig.add(x, sol, label="solution")

In [ ]:
fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/jdd2.dt_ev",  label=" ")
fig.scale("linear", "log")

fig = plot.Graph()
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd2_DX.son", marker="-x", label="TRUST-x", compo=0)
x = np.linspace(0, 1, 100)
sol = x*x
fig.add(x, sol, label="solution-x")


In [ ]:
fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/jdd3.dt_ev",  label=" ")
fig.scale("linear", "log")

fig = plot.Graph()
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd3_DY.son", marker="-x", label="TRUST-x", compo=0)
fig.addSegment(f"{run.BUILD_DIRECTORY}/jdd3_DY.son", marker="-x", label="TRUST-y", compo=1)
coef = 1000 * 10 / 2 / 10
y = np.linspace(0, 1, 100)
sol = coef * (y * y - 1)
fig.add(y, 0*sol, label="solution-x")
fig.add(y, sol, label="solution-y")

In [ ]:
fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/results/jdd4.dt_ev",  label=" ")
fig.scale("linear", "log")

fig = plot.Graph()
fig.addSegment(f"{run.BUILD_DIRECTORY}/results/jdd4_DX.son", marker="x", label="TRUST-x", compo=0)
fig.addSegment(f"{run.BUILD_DIRECTORY}/results/jdd4_DX.son", marker="x", label="TRUST-y", compo=1)
x = np.linspace(0, 1, 100)
ux = A*x
fig.add(x, ux, label="solution-x")
fig.add(x, 0*ux, label="solution-y")
# fig.add(y, sol, label="solution-y")